# Exercise: An Interactive Chat Loop with Claude

This exercise wraps the **message-list pattern** from the earlier notebooks in a small **REPL** (read-eval-print loop). You type a question, Claude answers, and the conversation history grows on each turn so Claude keeps full context.

## Attribution

Adapted with thanks from **[jaozc/building-with-the-claude-api](https://github.com/jaozc/building-with-the-claude-api/tree/main)**.

Changes for this course: swapped `%pip install` for the **uv** install pattern (uv-managed venvs ship without pip), set the model to **`claude-haiku-4-5`** (the repo default), and rewrote cloud-specific prompts to **Azure**.

### 1. Install dependencies

In [ ]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

### 2. Load environment variables from .env file

`load_dotenv()` reads your **`.env`** file so `ANTHROPIC_API_KEY` is available to the client. The key is never hardcoded.

In [ ]:
# Load environment variables
from dotenv import load_dotenv

load_dotenv()

### 3. Create an API client

`Anthropic()` reads the key from the environment. The `model` is pinned to **`claude-haiku-4-5`**, the course default, fast and inexpensive for a back-and-forth chat.

In [ ]:
# Import the Anthropic client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"  # unversioned alias -> current Haiku 4.5 snapshot

### 4. Helper functions

Three small helpers keep the loop readable. `add_user_message` and `add_assistant_message` **append** the correct `role` to the running `messages` list, and `chat` sends that whole list to the API and returns the reply text. Passing the **full history** on every call is what gives Claude memory of the conversation.

| Helper | Role written | What it does |
| --- | --- | --- |
| `add_user_message` | `user` | Appends your turn to the history |
| `add_assistant_message` | `assistant` | Appends Claude's reply to the history |
| `chat` | (none) | Sends the full history, returns reply text |

In [ ]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return message.content[0].text

### 5. Run the interactive chat loop

Now wrap the helpers in a **REPL**. Each pass reads one line with `input()`, appends it as a **user** turn, calls `chat`, then appends Claude's reply as an **assistant** turn. Because the history keeps growing, every reply has full context from earlier turns.

The **guard** is the load-bearing part of this exercise. Before sending, the loop checks for an **empty line** or an explicit **`quit`** / **`exit`** and breaks cleanly instead of calling the API.

> **Gotcha:** the Messages API rejects an **empty user turn** with a **400** (`user messages must have non-empty content`). Always guard `input()` so a stray Enter cannot send `""`.

To run it: execute the cell, type a question at the `>` prompt, and press Enter. Press Enter on an empty line, or type `quit` or `exit`, to end the session.

In [ ]:
messages = []

# Interactive chat loop. Type a question and press Enter.
# Press Enter on an empty line (or type "quit"/"exit") to end the chat.
# Why the guard: the Messages API rejects an empty user turn with a 400
# ("user messages must have non-empty content"), so we must not send "".
while True:
    user_input = input("> ").strip()

    # Empty line or an explicit quit word ends the session cleanly.
    if user_input == "" or user_input.lower() in {"quit", "exit"}:
        print("[chat ended]")
        break

    print(user_input)

    add_user_message(messages, user_input)
    response = chat(messages)
    add_assistant_message(messages, response)

    print(response)
